# Keyword Spotting Model Training
## Based on MF2143 Tutorial 2 - Local Training Workflow

This notebook replicates the Colab training workflow locally for training a keyword spotting model on the Speech Commands dataset.

**Requirements:**
- **Python 3.11 REQUIRED** - This has been tested and verified to work with Python 3.11 only
- GPU with CUDA support is recommended for faster and more memory-safe training
- **CPU training warning:** the full, unchanged workload can exhaust system RAM and swap on machines with about 16 GB RAM. Close memory-heavy applications, monitor `free -h`, and reduce `BATCH_SIZE` if GPU execution is unavailable.


In [ ]:
# Verify the training environment before importing TensorFlow.
import os
import shutil
import sys
import warnings

# Use a controlled CPU baseline by default. Set this to True only when the
# workstation has a verified NVIDIA/CUDA configuration.
USE_GPU = False
if not USE_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

missing_audio_tools = [
    tool for tool in ("ffmpeg", "ffprobe") if shutil.which(tool) is None
]
if missing_audio_tools:
    raise RuntimeError(
        "Missing system audio tools: "
        + ", ".join(missing_audio_tools)
        + ". On Ubuntu, install them with: sudo apt install ffmpeg"
    )

print(f"Python: {sys.version}")
print(f"Python executable: {sys.executable}")
print(f"Execution mode: {'GPU requested' if USE_GPU else 'CPU baseline'}")
print(f"FFmpeg: {shutil.which('ffmpeg')}")
print(f"FFprobe: {shutil.which('ffprobe')}")

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if USE_GPU:
    if not gpus:
        raise RuntimeError(
            "GPU execution was requested, but TensorFlow detected no GPU. "
            "Install the CUDA dependencies with: "
            "python -m pip install --upgrade 'tensorflow[and-cuda]==2.20.0'"
        )
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    warnings.warn(
        "CPU training is enabled. The full workload can exhaust system RAM "
        "and swap on machines with about 16 GB RAM. Close memory-heavy "
        "applications, monitor 'free -h', and reduce BATCH_SIZE if needed.",
        RuntimeWarning,
    )

print(f"TensorFlow: {tf.__version__}")
print(f"Detected GPUs: {gpus}")

assert sys.version_info[:2] == (3, 11), (
    f"Wrong Python version. Expected 3.11, got {sys.version_info[:2]}"
)
print("Correct training environment detected.")


## Step 1: Download and Load the Speech Commands Dataset

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import pathlib

# TFDS stores the downloaded and prepared dataset under this directory.
DATASET_DIR = pathlib.Path.cwd() / "datasets"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Three commands are learned explicitly. TFDS-provided silence and unknown
# examples are retained as a fourth class.
TARGET_WORDS = ["left", "right", "go"]
AUXILIARY_WORDS = ["_silence_", "_unknown_"]

print(f"TFDS directory: {DATASET_DIR}")
print(f"Target keywords: {TARGET_WORDS}")

# Download and prepare Speech Commands once in TFDS format. Subsequent runs
# reuse the prepared data under datasets/speech_commands/.
print("\nLoading Speech Commands with TensorFlow Datasets...")
builder = tfds.builder("speech_commands", data_dir=str(DATASET_DIR))
builder.download_and_prepare()
dataset_info = builder.info

ds_train = builder.as_dataset(split="train", shuffle_files=True)
ds_val = builder.as_dataset(split="validation", shuffle_files=False)
ds_test = builder.as_dataset(split="test", shuffle_files=False)

label_names = dataset_info.features["label"].names
selected_names = TARGET_WORDS + AUXILIARY_WORDS
missing_names = [name for name in selected_names if name not in label_names]
if missing_names:
    raise ValueError(f"Labels not provided by TFDS: {missing_names}")

SELECTED_LABEL_IDS = tf.constant(
    [label_names.index(name) for name in selected_names],
    dtype=tf.int64,
)

def keep_selected_label(example):
    """Keep target commands plus the TFDS silence and unknown classes."""
    return tf.reduce_any(tf.equal(example["label"], SELECTED_LABEL_IDS))

ds_train = ds_train.filter(keep_selected_label)
ds_val = ds_val.filter(keep_selected_label)
ds_test = ds_test.filter(keep_selected_label)

print("Dataset loaded and filtered successfully.")
print("Selected labels:")
for name in selected_names:
    print(f"  {name}: TFDS label {label_names.index(name)}")


In [ ]:
# Check the sizes of the filtered dataset splits.
print("\n" + "=" * 60)
print("FILTERED DATASET SIZE STATISTICS")
print("=" * 60)

def count_examples(dataset):
    """Count examples after tf.data filtering."""
    count = dataset.reduce(
        tf.constant(0, dtype=tf.int64),
        lambda total, _: total + 1,
    )
    return int(count.numpy())

train_size = count_examples(ds_train)
val_size = count_examples(ds_val)
test_size = count_examples(ds_test)
total = train_size + val_size + test_size

print("\nNumber of selected examples:")
print(f"  Training set:   {train_size:,}")
print(f"  Validation set: {val_size:,}")
print(f"  Test set:       {test_size:,}")
print(f"  Total:          {total:,}")

print("\nTFDS information:")
print(f"  Dataset name: {dataset_info.name}")
print(f"  Version: {dataset_info.version}")
print(f"  Full download size: {dataset_info.download_size / (1024**2):.2f} MB")
print(f"  Full prepared size: {dataset_info.dataset_size / (1024**2):.2f} MB")

print("\nFiltered split percentages:")
print(f"  Training:   {(train_size / total) * 100:.1f}%")
print(f"  Validation: {(val_size / total) * 100:.1f}%")
print(f"  Test:       {(test_size / total) * 100:.1f}%")
print("\n" + "=" * 60)


## Step 2: Explore Dataset Structure

In [ ]:
# Explore one filtered example.
for example in ds_train.take(1):
    label_id = int(example["label"].numpy())
    print("Keys:", list(example.keys()))
    print("Audio shape:", example["audio"].shape)
    print("Audio dtype:", example["audio"].dtype)
    print("TFDS label:", label_id)
    print("Label name:", label_names[label_id])

print("\nAll TFDS labels:")
print(label_names)
print(f"\nLabels retained for this tutorial: {selected_names}")


## Step 3: Configure Target Keywords

As per the tutorial, we'll train on three navigation commands: **left**, **right**, and **go**

In [ ]:
# Configure keywords and audio parameters
WANTED_WORDS = TARGET_WORDS
NUM_CLASSES = 4  # left, right, go, unknown (silence merged with unknown)

# Audio parameters (matching TFLite Micro micro_speech)
SAMPLE_RATE = 16000
CLIP_DURATION_MS = 2000  # 2 second clips
WINDOW_SIZE_MS = 25.0    # 30ms window
WINDOW_STRIDE_MS = 10.0  # 20ms stride
FEATURE_BIN_COUNT = 32   # Number of frequency bins

# Training parameters
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.001

# Data augmentation parameters
BACKGROUND_NOISE_VOLUME = 0.1
TIME_SHIFT_MS = 100.0

# Calculate spectrogram parameters
WINDOW_SIZE_SAMPLES = int(SAMPLE_RATE * WINDOW_SIZE_MS / 1000)
WINDOW_STRIDE_SAMPLES = int(SAMPLE_RATE * WINDOW_STRIDE_MS / 1000)
LENGTH_MINUS_WINDOW = (SAMPLE_RATE * CLIP_DURATION_MS / 1000) - WINDOW_SIZE_SAMPLES
SPECTROGRAM_LENGTH = int(LENGTH_MINUS_WINDOW / WINDOW_STRIDE_SAMPLES) + 1

print(f"Target keywords: {WANTED_WORDS}")
print(f"Label mapping: left=0, right=1, go=2, unknown/silence=3")
print(f"\nSpectrogram parameters:")
print(f"  Window size: {WINDOW_SIZE_SAMPLES} samples ({WINDOW_SIZE_MS}ms)")
print(f"  Window stride: {WINDOW_STRIDE_SAMPLES} samples ({WINDOW_STRIDE_MS}ms)")
print(f"  Frequency bins: {FEATURE_BIN_COUNT}")
print(f"  Time steps: {SPECTROGRAM_LENGTH}")
print(f"  Input shape: ({SPECTROGRAM_LENGTH}, {FEATURE_BIN_COUNT})")

## Step 4: Preprocess and Filter Dataset

In [ ]:

def preprocess_audio_to_spectrogram(example):
    """Convert audio to log mel spectrogram"""
    audio = example['audio']
    label = example['label']
    
    # Normalize audio to [-1, 1]
    audio = tf.cast(audio, tf.float32) / 32768.0
    
    # Pad or trim to exactly 1 second (16000 samples)
    audio_length = tf.shape(audio)[0]
    if_short = lambda: tf.concat([audio, tf.zeros(32000 - audio_length)], axis=0)
    if_long = lambda: audio[:32000]
    audio = tf.cond(audio_length < 32000, if_short, if_long)
    audio = tf.reshape(audio, [32000])
    
    # Compute STFT (Short-Time Fourier Transform)
    stft = tf.signal.stft(
        audio,
        frame_length=WINDOW_SIZE_SAMPLES,
        frame_step=WINDOW_STRIDE_SAMPLES,
        fft_length=WINDOW_SIZE_SAMPLES
    )
    
    # Get magnitude spectrogram
    spectrogram = tf.abs(stft)
    
    # Convert to mel scale
    # Create mel filterbank
    num_spectrogram_bins = spectrogram.shape[-1]
    lower_edge_hertz, upper_edge_hertz = 20.0, 4000.0
    
    linear_to_mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        FEATURE_BIN_COUNT,
        num_spectrogram_bins,
        SAMPLE_RATE,
        lower_edge_hertz,
        upper_edge_hertz
    )
    
    mel_spectrogram = tf.tensordot(spectrogram, linear_to_mel_weight_matrix, 1)
    
    # Convert to log scale
    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)
    
    # Ensure fixed shape
    log_mel_spectrogram = tf.ensure_shape(log_mel_spectrogram, [SPECTROGRAM_LENGTH, FEATURE_BIN_COUNT])
    
    return log_mel_spectrogram, label

def add_background_noise(spectrogram, label):
    """Add random noise augmentation"""
    if tf.random.uniform([]) < 0.8:  # 80% chance to add noise
        noise = tf.random.normal(tf.shape(spectrogram), mean=0.0, stddev=BACKGROUND_NOISE_VOLUME)
        spectrogram = spectrogram + noise
    return spectrogram, label

def time_shift(audio_data, label, shift_ms=100.0):
    """Randomly shift audio in time"""
    shift_amount = int(SAMPLE_RATE * shift_ms / 1000)
    shift = tf.random.uniform([], -shift_amount, shift_amount, dtype=tf.int32)
    shifted_audio = tf.roll(audio_data, shift, axis=0)
    return shifted_audio, label

# Build the label table once. Target commands map to classes 0-2;
# silence and all unknown speech map to class 3.
target_label_ids = [label_names.index(word) for word in WANTED_WORDS]
label_table = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(target_label_ids, dtype=tf.int64),
        values=tf.range(len(WANTED_WORDS), dtype=tf.int64),
    ),
    default_value=tf.constant(3, dtype=tf.int64),
)

def relabel_example(spectrogram, label):
    """Map TFDS labels to left=0, right=1, go=2, unknown/silence=3."""
    return spectrogram, label_table.lookup(label)

# Apply preprocessing to datasets
print("Preprocessing training dataset with augmentation...")
ds_train_processed = (
    ds_train
    .map(preprocess_audio_to_spectrogram, num_parallel_calls=tf.data.AUTOTUNE)
    .map(add_background_noise, num_parallel_calls=tf.data.AUTOTUNE)
    .map(relabel_example, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    # Filtering hides the finite size from tf.data. Restore the known number
    # of batches so Keras does not report "Your input ran out of data".
    .apply(tf.data.experimental.assert_cardinality(
        (train_size + BATCH_SIZE - 1) // BATCH_SIZE
    ))
    .prefetch(tf.data.AUTOTUNE)
)

print("Preprocessing validation dataset (no augmentation)...")
ds_val_processed = (
    ds_val
    .map(preprocess_audio_to_spectrogram, num_parallel_calls=tf.data.AUTOTUNE)
    .map(relabel_example, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .apply(tf.data.experimental.assert_cardinality(
        (val_size + BATCH_SIZE - 1) // BATCH_SIZE
    ))
    .prefetch(tf.data.AUTOTUNE)
)

print("Preprocessing test dataset (no augmentation)...")
ds_test_processed = (
    ds_test
    .map(preprocess_audio_to_spectrogram, num_parallel_calls=tf.data.AUTOTUNE)
    .map(relabel_example, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .apply(tf.data.experimental.assert_cardinality(
        (test_size + BATCH_SIZE - 1) // BATCH_SIZE
    ))
    .prefetch(tf.data.AUTOTUNE)
)

print("\nDataset preprocessing complete!")
print(f"Training batches: {tf.data.experimental.cardinality(ds_train_processed)}")
print(f"Validation batches: {tf.data.experimental.cardinality(ds_val_processed)}")
print(f"Test batches: {tf.data.experimental.cardinality(ds_test_processed)}")

## Step 5: Visualize Spectrogram Examples

Let's visualize what the spectrograms look like for each of our target keywords.

In [ ]:
# Visualize spectrograms for each keyword
fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 8),
    constrained_layout=True,
)
axes = axes.flatten()

keyword_names = ["left", "right", "go", "unknown"]
examples_found = {i: 0 for i in range(4)}

for spectrograms, labels in ds_train_processed.take(50):
    for i in range(len(labels)):
        label = int(labels[i].numpy())
        if label < 4 and examples_found[label] == 0:
            ax = axes[label]
            spectrogram = spectrograms[i].numpy()
            im = ax.imshow(
                spectrogram.T,
                aspect="auto",
                origin="lower",
                cmap="viridis",
            )
            ax.set_title(f"{keyword_names[label]} (class {label})")
            ax.set_xlabel("Time")
            ax.set_ylabel("Frequency Bin")
            examples_found[label] = 1

    if all(examples_found.values()):
        break

if not all(examples_found.values()):
    missing = [
        keyword_names[label]
        for label, found in examples_found.items()
        if not found
    ]
    raise RuntimeError(f"No visualization example found for: {missing}")

fig.colorbar(
    im,
    ax=axes.tolist(),
    label="Log Mel Energy",
)

fig.savefig(
    "spectrogram_examples.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

print("Spectrogram visualization saved to 'spectrogram_examples.png'")


## Step 6: Build CNN Model for Spectrogram Input

This model is based on the TFLite Micro tiny_conv architecture, optimized for spectrogram inputs.

In [ ]:
def build_tiny_conv_model(input_shape=(SPECTROGRAM_LENGTH, FEATURE_BIN_COUNT), num_classes=4):
    """Build tiny_conv model similar to TFLite Micro micro_speech"""
    
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        
        # Expand dimensions for Conv2D
        tf.keras.layers.Reshape((input_shape[0], input_shape[1], 1)),
        
        # First conv block - larger filters to capture temporal-frequency patterns
        tf.keras.layers.Conv2D(8, (10, 8), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),
        
        # Second conv block
        tf.keras.layers.Conv2D(16, (5, 4), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),
        
        # Third conv block
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Dropout(0.25),
        
        # Flatten and classify
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Build the model
model = build_tiny_conv_model(num_classes=NUM_CLASSES)
model.summary()

# Print input/output shapes
print(f"\nModel expects input shape: {model.input_shape}")
print(f"Model output shape: {model.output_shape}")

## Step 7: Compile the Model

In [ ]:
# Calculate class weights to handle imbalance
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Collect labels from training set to compute class weights
print("Computing class weights from training data...")
all_labels = []
for _, labels in ds_train_processed.unbatch().batch(1000).take(100):
    all_labels.extend(labels.numpy())

all_labels = np.array(all_labels)
unique_classes = np.unique(all_labels)
class_weights_array = compute_class_weight('balanced', classes=unique_classes, y=all_labels)
class_weights = dict(zip(unique_classes.astype(int), class_weights_array))

print(f"\nClass weights: {class_weights}")
print("Higher weights = more penalty for misclassification")

# Compile the model with learning rate schedule
initial_learning_rate = LEARNING_RATE

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=initial_learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel compiled successfully!")
print(f"Initial learning rate: {initial_learning_rate}")
print("\nClass weights will be applied during training to handle imbalance.")

## Step 8: Train the Model

Training parameters:
- **Epochs**: 30 (to ensure convergence with spectrogram features)
- **Batch size**: 64
- **Learning rate**: 0.001 with decay
- **Data augmentation**: Background noise (80% probability)

Expected outcome: validation accuracy above 90% for keyword spotting.

In [ ]:
# Define callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir='logs',
        histogram_freq=1
    )
]

# Train the model
print("Starting training...")
print(f"Total epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Training samples: ~85,000")
print(f"Validation samples: ~10,000\n")

history = model.fit(
    ds_train_processed,
    validation_data=ds_val_processed,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Model Accuracy')
ax1.legend()
ax1.grid(True)

# Loss plot
ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Model Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

# Print final results
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f"\nFinal Training Accuracy: {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")

## Step 9: Evaluate on Test Set

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_loss, test_accuracy = model.evaluate(ds_test_processed, verbose=1)
print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

## Step 10: Export TensorFlow Lite Models

We'll export both:
1. **Float32 model** - Full precision model
2. **INT8 quantized model** - Required for Edge TPU deployment

In [ ]:
import os
import numpy as np

# Create models directory
os.makedirs('trained_models', exist_ok=True)

print("=" * 60)
print("Saving Model and Calibration Data")
print("=" * 60)

# Step 1: Save the trained model
print("\n1. Saving Keras model...")
model_path = 'trained_models/model.keras'
model.save(model_path)
print(f"   ✓ Saved: {model_path}")

# Step 2: Save calibration data for INT8 quantization
print("\n2. Collecting calibration data...")
calibration_samples = []
for i, (spectrograms, _) in enumerate(ds_train_processed.take(50)):
    calibration_samples.append(spectrograms[0].numpy())
    if i % 10 == 0:
        print(f"   Collecting batch {i}/50...")

calibration_data = np.array(calibration_samples)
calibration_path = 'trained_models/calibration_data.npy'
np.save(calibration_path, calibration_data)
print(f"   ✓ Saved {len(calibration_data)} samples to: {calibration_path}")

print("\n" + "=" * 60)
print("✓ Model and calibration data saved!")
print("=" * 60)
print("\nYou can now restart the kernel and run the next cell")
print("to convert to TFLite without retraining.")

In [ ]:
import os
import tensorflow as tf
import numpy as np

# Paths
model_path = 'trained_models/model.keras'
calibration_path = 'trained_models/calibration_data.npy'
float_output = 'trained_models/model.tflite'
int8_output = 'trained_models/model_int8.tflite'

print("=" * 60)
print("TFLite Model Conversion")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")

# Check if files exist
if not os.path.exists(model_path):
    print(f"\n✗ Error: Model not found at {model_path}")
    print("Please run the previous cell to save the model first.")
else:
    print(f"\n✓ Found model: {model_path}")

if not os.path.exists(calibration_path):
    print(f"✗ Error: Calibration data not found at {calibration_path}")
else:
    print(f"✓ Found calibration data: {calibration_path}")

# Load model
print("\nLoading model...")
loaded_model = tf.keras.models.load_model(model_path)
print("✓ Model loaded")
print(f"Model input shape: {loaded_model.input_shape}")
print(f"Model output shape: {loaded_model.output_shape}")

# Load calibration data
calibration_data = np.load(calibration_path)
print(f"\nLoaded {len(calibration_data)} calibration samples")

# Create concrete function with fixed batch size
run_model = tf.function(lambda x: loaded_model(x))
concrete_func = run_model.get_concrete_function(
    tf.TensorSpec(shape=[1, 198, 32], dtype=tf.float32)
)

# ============================================================
# Step 1: Float32 TFLite with fixed input shape
# ============================================================
print("\n" + "=" * 60)
print("1. Converting to Float32 TFLite")
print("=" * 60)

converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()

with open(float_output, 'wb') as f:
    f.write(tflite_model)

float_size = os.path.getsize(float_output) / 1024
print(f"✓ Float32 model saved: {float_output}")
print(f"✓ Size: {float_size:.2f} KB")

# ============================================================
# Step 2: Full INT8 TFLite (TFLite Micro compatible)
# ============================================================
print("\n" + "=" * 60)
print("2. Converting to Full INT8 TFLite")
print("=" * 60)

# Representative dataset generator
def representative_dataset_gen():
    for i in range(len(calibration_data)):
        if i % 10 == 0:
            print(f"  Processing sample {i}/{len(calibration_data)}")
        sample = np.expand_dims(calibration_data[i], axis=0).astype(np.float32)
        yield [sample]

print("\nConverting with full INT8 quantization (TFLite Micro compatible)...")
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen

# CRITICAL: Set input/output types to INT8 for TFLite Micro compatibility
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

try:
    tflite_quant_model = converter.convert()
    
    with open(int8_output, 'wb') as f:
        f.write(tflite_quant_model)
    
    int8_size = os.path.getsize(int8_output) / 1024
    compression = (1 - int8_size / float_size) * 100
    
    print(f"\n✓ Full INT8 model saved: {int8_output}")
    print(f"✓ Size: {int8_size:.2f} KB")
    print(f"✓ Compression: {compression:.1f}% reduction")
    print(f"✓ TFLite Micro compatible (no QUANTIZE/DEQUANTIZE ops)")
    
    print("\n" + "=" * 60)
    print("Next Steps:")
    print("=" * 60)
    print("\n1. Compile with Edge TPU Compiler:")
    print(f"   edgetpu_compiler {int8_output}")
    print("\n2. Copy compiled model:")
    print("   cp model_int8_edgetpu.tflite models/")
    print("\n3. Rebuild and flash:")
    print("   bash build.sh")
    print("   python3 scripts/flashtool.py --build_dir build \\")
    print("     --elf_path build/examples/classify_keywords/classify_keywords")
    
except Exception as e:
    print(f"\n✗ Full INT8 conversion failed: {type(e).__name__}")
    print(f"Error: {str(e)[:300]}")
    print("\nThis may be due to TensorFlow version compatibility.")
    print("Try using the Float32 model and let Edge TPU Compiler quantize it:")
    print(f"  edgetpu_compiler {float_output}")

print("\n" + "=" * 60)

## Checkpoint Questions (from Tutorial)

1. **What final accuracy did your model achieve?**
   - Check the test accuracy above

2. **Compare file sizes of float vs INT8 TFLite models.**
   - See the size comparison above

3. **Why is INT8 quantization required for Coral Edge TPU?**
   - Edge TPU is optimized for INT8 operations, providing faster inference with lower power consumption
   - INT8 models are much smaller, fitting better in limited memory
   - The hardware accelerator only supports quantized operations

4. **How do the left, right, and go commands map to robot motion?**
   - left (0): Turn robot left
   - right (1): Turn robot right  
   - go (2): Move robot forward

## Summary

You have successfully:
- ✅ Downloaded the Speech Commands dataset
- ✅ Configured for three keywords (left, right, go)
- ✅ Built and trained a CNN keyword spotting model
- ✅ Exported both Float32 and INT8 TFLite models
- ✅ Documented training results

**Next Steps:**
1. Compile the INT8 model for Edge TPU
2. Deploy to Coral Dev Board Micro
3. Integrate with MATLAB simulation (Task 3)
4. Conduct Hardware-in-the-Loop testing (Task 4)